# Optimizer comparison on classification using Digits dataset and Logistic Regression

In [28]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor, Lambda, Compose

from src.quadratic_annealing_optimizer import QuadraticAnnealingOptimizer
from src.models import Logistic
from src.utils import data_load_and_prep, build_sampler
from src.training import train
from src.newton_optimizer import NewtonOptimizer

EXPERIMENT_NAME = "optimizer-comparison-logreg-MNIST-q"

In [29]:
# Loading MNIST dataset
transform = Compose([
    ToTensor(),
    Lambda(lambda x: x.view(-1)),
])
train_data = datasets.MNIST(root="../data", train=True, download=False, transform=transform)
test_data = datasets.MNIST(root="../data", train=False, download=False, transform=transform)

In [30]:
NUM_EPOCHS = 30
# BATH_SIZE = 128
train_loader = DataLoader(train_data, batch_size=len(train_data), shuffle=True)
test_loader = DataLoader(test_data, batch_size=len(test_data), shuffle=False)

In [31]:
X, y = next(iter(train_loader))
X.shape, y.shape

(torch.Size([60000, 784]), torch.Size([60000]))

## Adam

In [32]:
loss_fn = nn.CrossEntropyLoss()
adam_model = Logistic(input_dim=784, output_dim=10)
classical_device = "cpu" 
adam_optimizer = torch.optim.Adam(adam_model.parameters(), 
                             lr=0.01,
                             betas=[0.9, 0.999],
                             )

## Adam optimizaer

In [33]:
train(
    model=adam_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=adam_optimizer,
    c_device=classical_device,
    epochs=NUM_EPOCHS, 
    experiment_name = EXPERIMENT_NAME,
    run_name="adam-optimizer"
)

Epoch 000 | train_loss=1.8867 | train_acc=0.539 | test_loss=1.8787 | test_acc=0.552 | 
Epoch 005 | train_loss=0.8244 | train_acc=0.809 | test_loss=0.8008 | test_acc=0.817 | 
Epoch 010 | train_loss=0.5655 | train_acc=0.846 | test_loss=0.5406 | test_acc=0.858 | 
Epoch 015 | train_loss=0.4680 | train_acc=0.868 | test_loss=0.4462 | test_acc=0.878 | 
Epoch 020 | train_loss=0.4209 | train_acc=0.880 | test_loss=0.4021 | test_acc=0.890 | 
Epoch 025 | train_loss=0.3906 | train_acc=0.889 | test_loss=0.3728 | test_acc=0.897 | 


2026/04/27 16:15:42 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 029 | train_loss=0.3734 | train_acc=0.894 | test_loss=0.3572 | test_acc=0.902 | 


{'run_id': 'cf347180ef2e49e793bd7d72043b1822',
 'experiment_id': '670551355725287584',
 'experiment_name': 'optimizer-comparison-logreg-MNIST-q',
 'tracking_uri': 'file:///home/filip/studia/master/second-order-by-annealer/mlruns',
 'optimizer_time_sec': 1.1882668100079172,
 'train_evaluation_time_sec': 115.59232396099469,
 'test_evaluation_time_sec': 18.249987894989317,
 'evaluation_time_sec': 133.842311855984,
 'mlflow_logging_time_sec': 0.08345511400329997,
 'training_time_sec': 250.9412789329981,
 'final_train_loss': 0.3733934462070465,
 'final_test_loss': 0.3572354018688202,
 'final_train_metric': 0.89385,
 'final_test_metric': 0.9018,
 'optimizer': 'Adam',
 'seed': None}

## l-BFGS optimizer

In [6]:
loss_fn = nn.CrossEntropyLoss()
lbfgs_model = Logistic(784, 10)
classical_device = "cuda" 
lbfgs_optimizer = torch.optim.LBFGS(lbfgs_model.parameters(), 
                              lr=0.01,
                              max_iter=1,
                              history_size=100,
                              line_search_fn=None,
                              )

In [7]:
train(
    model=lbfgs_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=lbfgs_optimizer,
    c_device=classical_device,
    epochs=NUM_EPOCHS,
    experiment_name = EXPERIMENT_NAME,
    run_name="lbfgs-optimizer"
)

Epoch 000 | train_loss=2.2975 | train_acc=0.153 | test_loss=2.2972 | test_acc=0.159 | 
Epoch 005 | train_loss=2.1100 | train_acc=0.333 | test_loss=2.1063 | test_acc=0.340 | 
Epoch 010 | train_loss=1.9168 | train_acc=0.624 | test_loss=1.9093 | test_acc=0.629 | 
Epoch 015 | train_loss=1.7587 | train_acc=0.756 | test_loss=1.7482 | test_acc=0.757 | 
Epoch 020 | train_loss=1.6310 | train_acc=0.803 | test_loss=1.6181 | test_acc=0.808 | 
Epoch 025 | train_loss=1.5178 | train_acc=0.827 | test_loss=1.5029 | test_acc=0.836 | 


2026/04/26 21:05:09 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 029 | train_loss=1.4460 | train_acc=0.837 | test_loss=1.4300 | test_acc=0.847 | 


{'run_id': 'aeee758f86ce4067b6ad04569a4071a4',
 'experiment_id': '670551355725287584',
 'experiment_name': 'optimizer-comparison-logreg-MNIST-q',
 'tracking_uri': 'file:///home/filip/studia/master/second-order-by-annealer/mlruns',
 'optimizer_time_sec': 0.10596142298163613,
 'training_time_sec': 252.20047761499882,
 'final_train_loss': 1.445955753326416,
 'final_test_loss': 1.4299848079681396,
 'final_train_metric': 0.8371,
 'final_test_metric': 0.8473,
 'optimizer': 'LBFGS',
 'seed': None}

## Newton-Raphson optimizer

In [8]:
loss_fn = nn.CrossEntropyLoss()
newton_model = Logistic(784, 10)
classical_device = "cuda" 
newton_optimizer = NewtonOptimizer(newton_model.parameters(), 
                              lr=0.1,
                              max_iter=1,
                              damping=1e-4,
                              )

In [9]:
train(
    model=newton_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=newton_optimizer,
    c_device=classical_device,
    epochs=NUM_EPOCHS,
    experiment_name = EXPERIMENT_NAME,
    run_name="newton-optimizer"
)

Epoch 000 | train_loss=1.8255 | train_acc=0.791 | test_loss=1.8245 | test_acc=0.793 | 
Epoch 005 | train_loss=1.0071 | train_acc=0.892 | test_loss=1.0021 | test_acc=0.891 | 
Epoch 010 | train_loss=0.6900 | train_acc=0.905 | test_loss=0.6873 | test_acc=0.904 | 
Epoch 015 | train_loss=0.5155 | train_acc=0.914 | test_loss=0.5168 | test_acc=0.911 | 
Epoch 020 | train_loss=0.4093 | train_acc=0.920 | test_loss=0.4151 | test_acc=0.915 | 
Epoch 025 | train_loss=0.3415 | train_acc=0.925 | test_loss=0.3522 | test_acc=0.919 | 


2026/04/26 21:12:12 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 029 | train_loss=0.3049 | train_acc=0.929 | test_loss=0.3196 | test_acc=0.920 | 


{'run_id': '6896e441f36643e49c9144e4ec350f4f',
 'experiment_id': '670551355725287584',
 'experiment_name': 'optimizer-comparison-logreg-MNIST-q',
 'tracking_uri': 'file:///home/filip/studia/master/second-order-by-annealer/mlruns',
 'optimizer_time_sec': 167.7254191900065,
 'training_time_sec': 421.2048856610054,
 'final_train_loss': 0.30486324429512024,
 'final_test_loss': 0.319551020860672,
 'final_train_metric': 0.9285333333333333,
 'final_test_metric': 0.9203,
 'optimizer': 'NewtonOptimizer',
 'seed': None}

## Quadriatic Annealing optimizer (cpu)

In [4]:
loss_fn = nn.CrossEntropyLoss()
model = Logistic(784, 10)
classical_device = "cpu"

sampler = build_sampler(mode="simulated")

optimizer = QuadraticAnnealingOptimizer(
    sampler=sampler,
    model=model,
    device="cpu",
    subset_size=12,
    step_size=0.05,
    num_reads=100,
)

In [ ]:
train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    c_device=classical_device,
    epochs=NUM_EPOCHS,
    experiment_name = EXPERIMENT_NAME,
    run_name="quadratic-annealer-cpu-optimizer"
)

/home/filip/studia/master/second-order-by-annealer/venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


Epoch 000 | train_loss=2.2476 | train_acc=0.173 | test_loss=2.2493 | test_acc=0.177 | 
Epoch 005 | train_loss=2.1166 | train_acc=0.311 | test_loss=2.1189 | test_acc=0.309 | 
Epoch 010 | train_loss=2.0052 | train_acc=0.406 | test_loss=2.0079 | test_acc=0.409 | 
Epoch 015 | train_loss=1.9040 | train_acc=0.482 | test_loss=1.9072 | test_acc=0.486 | 
Epoch 020 | train_loss=1.8107 | train_acc=0.523 | test_loss=1.8114 | test_acc=0.528 | 
Epoch 025 | train_loss=1.7268 | train_acc=0.543 | test_loss=1.7258 | test_acc=0.549 | 
Epoch 030 | train_loss=1.6452 | train_acc=0.589 | test_loss=1.6429 | test_acc=0.593 | 
Epoch 035 | train_loss=1.5738 | train_acc=0.604 | test_loss=1.5720 | test_acc=0.605 | 
Epoch 040 | train_loss=1.5036 | train_acc=0.626 | test_loss=1.5001 | test_acc=0.625 | 
Epoch 045 | train_loss=1.4369 | train_acc=0.658 | test_loss=1.4320 | test_acc=0.662 | 


2026/04/02 09:03:45 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 049 | train_loss=1.3889 | train_acc=0.671 | test_loss=1.3826 | test_acc=0.676 | 


## Quadratic Annealing optimizer (cpu + momentum)

In [10]:
loss_fn = nn.CrossEntropyLoss()
model = Logistic(784, 10)
classical_device = "cpu"

sampler = build_sampler(mode="simulated")

optimizer = QuadraticAnnealingOptimizer(
    sampler=sampler,
    model=model,
    device="cpu",
    subset_size=12,
    step_size=0.1,
    num_reads=100,
    beta=0.9,
)

In [11]:
train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    c_device=classical_device,
    epochs=NUM_EPOCHS,
    experiment_name = EXPERIMENT_NAME,
    run_name="quadratic-annealer-cpu-momentum-optimizer"
)

Epoch 000 | train_loss=2.2350 | train_acc=0.198 | test_loss=2.2299 | test_acc=0.206 | 
Epoch 005 | train_loss=1.9602 | train_acc=0.386 | test_loss=1.9573 | test_acc=0.385 | 
Epoch 010 | train_loss=1.7109 | train_acc=0.537 | test_loss=1.7020 | test_acc=0.548 | 
Epoch 015 | train_loss=1.5063 | train_acc=0.565 | test_loss=1.4978 | test_acc=0.568 | 
Epoch 020 | train_loss=1.3507 | train_acc=0.658 | test_loss=1.3379 | test_acc=0.664 | 
Epoch 025 | train_loss=1.2209 | train_acc=0.696 | test_loss=1.2068 | test_acc=0.700 | 


2026/04/26 21:16:46 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 029 | train_loss=1.1494 | train_acc=0.717 | test_loss=1.1300 | test_acc=0.726 | 


{'run_id': '262ec263ab384e23a4dbb44e485a1fb5',
 'experiment_id': '670551355725287584',
 'experiment_name': 'optimizer-comparison-logreg-MNIST-q',
 'tracking_uri': 'file:///home/filip/studia/master/second-order-by-annealer/mlruns',
 'optimizer_time_sec': 13.175769615030731,
 'training_time_sec': 270.31052209000336,
 'final_train_loss': 1.1494156122207642,
 'final_test_loss': 1.1300398111343384,
 'final_train_metric': 0.7167,
 'final_test_metric': 0.7256,
 'optimizer': 'QuadraticAnnealingOptimizer',
 'seed': None}

## Quadratic Annealing optimizer (dwave + momentum)

In [ ]:
loss_fn = nn.CrossEntropyLoss()
model = Logistic(784, 10)
classical_device = "cpu"

sampler = build_sampler(mode="dwave")

optimizer = QuadraticAnnealingOptimizer(
    sampler=sampler,
    model=model,
    device="cpu",
    subset_size=12,
    step_size=0.1,
    num_reads=100,
    beta=0.9,
)

In [ ]:
train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    c_device=classical_device,
    epochs=NUM_EPOCHS,
    experiment_name = EXPERIMENT_NAME,
    run_name="quadratic-annealer-dwave-momentum-optimizer"
)

Epoch 000 | train_loss=2.3002 | train_acc=0.166 | test_loss=2.2934 | test_acc=0.172 | 
Epoch 005 | train_loss=2.0081 | train_acc=0.347 | test_loss=2.0047 | test_acc=0.352 | 
Epoch 010 | train_loss=1.7768 | train_acc=0.502 | test_loss=1.7743 | test_acc=0.498 | 
Epoch 015 | train_loss=1.5728 | train_acc=0.596 | test_loss=1.5656 | test_acc=0.597 | 
Epoch 020 | train_loss=1.5728 | train_acc=0.596 | test_loss=1.5656 | test_acc=0.597 | 
Epoch 025 | train_loss=1.5728 | train_acc=0.596 | test_loss=1.5656 | test_acc=0.597 | 
Epoch 030 | train_loss=1.5728 | train_acc=0.596 | test_loss=1.5656 | test_acc=0.597 | 
Epoch 035 | train_loss=1.5728 | train_acc=0.596 | test_loss=1.5656 | test_acc=0.597 | 
Epoch 040 | train_loss=1.5728 | train_acc=0.596 | test_loss=1.5656 | test_acc=0.597 | 
Epoch 045 | train_loss=1.5728 | train_acc=0.596 | test_loss=1.5656 | test_acc=0.597 | 


2026/03/30 16:31:50 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 049 | train_loss=1.5728 | train_acc=0.596 | test_loss=1.5656 | test_acc=0.597 | 
